In [ ]:
get_ipython().system('hostname')

In [ ]:
from photometry.models.baselines import LambertianModel
from photometry.fitting.least_sq import LeastSquaresFitter
from photometry.core.types import GeometryBatch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 
import pandas as pd
from pathlib import Path
import rasterio
import spiceypy as spice
import plotly.graph_objects as go
from scipy.optimize import curve_fit
import duckdb
import os
print(os.getcwd())

In [ ]:
# Set up paths and load phase-curve data for all phases

project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent



parquet_path_survey_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "survey"/"*.parquet"

parquet_path_hamo_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "hamo"/"*.parquet"

parquet_path_lamo_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "lamo"/"*.parquet"

binned_parquet_path_survey_gaskell_dsk256_110825 = project_root / "data" / "golden" / "survey_binned_dsk256_110825_range80.parquet"



dtm_path = project_root / "data" / "dtm" / "DTM_VESTA_93M.TIF"

dsk_path = project_root / "data" / "spice_kernels" / "vesta_gaskell_256_110825.bds"

In [ ]:
survey_df = pd.read_parquet(binned_parquet_path_survey_gaskell_dsk256_110825)
survey_df.head()

In [ ]:
df = survey_df.copy()
df = df[df["alpha_grid"] < 80]  

i_rad = np.deg2rad(df["mean_incidence"].to_numpy())
e_rad = np.deg2rad(df["mean_emission"].to_numpy())
mu0 = np.cos(i_rad)
mu = np.cos(e_rad)

In [ ]:
df["LS-factor"] = mu0 / (mu0 + mu)   # Lommel-Seeliger geometry factor
df["iof"] = df["mean_iof"]
df["weights"] = df["n_pixels"]

In [ ]:
df.groupby("alpha_grid")
df

In [ ]:
get_ipython().system('hostname')
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"


from photometry.models.baselines import LommelSeeligerModel
from photometry.fitting.least_sq import LeastSquaresFitter
from photometry.core.types import GeometryBatch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
import dataclasses
get_ipython().system('pip install rich')
from rich import print
import os
print(os.getcwd())

In [ ]:
geometry = GeometryBatch(
    incidence=i_rad,
    emission=e_rad,
    phase=np.zeros(len(df)),
)
observed = df["iof"].to_numpy()
# weights = 1/sigma convention (see fitting/least_sq.py); sigma = 1/sqrt(n_pixels)
weights = np.sqrt(df["n_pixels"].to_numpy())

model = LommelSeeligerModel()
fitter = LeastSquaresFitter()

result = fitter.fit(
    model=model,
    geometry=geometry,
    observed_reflectance=observed,
    weights=weights,
)

print("Fit success:", result.metadata["success"])
print("Fitted w:", result.fitted_parameters["w"])
# parameter_errors is a first-class FitResult field (1-sigma per parameter). If w rails
# at its upper bound (w=1.0, as seen with real Vesta data — see boundary_hits below),
# its error is reported as NaN rather than a spurious curvature-based number.
print("Parameter errors:", result.parameter_errors)
print("Boundary hits:", result.metadata["boundary_hits"])
if result.metadata["error_estimation_warning"]:
    print("Error estimation warning:", result.metadata["error_estimation_warning"])

model.parameters.update(result.fitted_parameters)
predicted = np.asarray(model.reflectance(geometry))
resid = observed - predicted
weighted_rms = np.sqrt(np.average(resid**2, weights=df["n_pixels"].to_numpy()))
print(f"Weighted RMS: {weighted_rms:.5f}")
